# NLP Assignment — N-gram Smoothing Techniques

Use the four n-gram language models (Unigram, Bigram, Trigram, Quadrigram) from the previous assignment and evaluate the following smoothing techniques on the development and test sets:

1. Interpolated Smoothing (Bigram, Trigram, Quadrigram)
2. Good-Turing Smoothing
3. Katz Backoff Smoothing
4. Stupid Backoff Smoothing
5. Kneser-Ney Smoothing

Lower perplexity indicates better language-model performance.


## 1. Imports


In [1]:
import re
import math
from collections import Counter
import pyarrow.parquet as pq
import pandas as pd


## 2. Configuration


In [2]:
TRAIN_FILE = "../ASS-4/train.parquet"
DEV_FILE = "../ASS-4/dev.parquet"
TEST_FILE = "../ASS-4/test.parquet"
K = 0.3
print("K =", K)


K = 0.3


## 3. Assignment-1 tokenizer


In [3]:
TOKEN_PATTERN = re.compile(r"""
    https?://[^\s]+|
    \b[\w.+-]+@[\w.-]+\.\w+\b|
    \d{1,2}[/-]\d{1,2}[/-]\d{2,4}|
    \d+\.\d+|
    [\u0C66-\u0C6F]+|
    \d+|
    [\u0C00-\u0C7F]+|
    [A-Za-z]+|
    \.{3}|
    [.,!?;:()"'\-]
""", re.VERBOSE)

def tokenize(sentence):
    return TOKEN_PATTERN.findall(sentence)


## 4. Load and verify datasets


In [4]:
train_pf = pq.ParquetFile(TRAIN_FILE)
dev_pf = pq.ParquetFile(DEV_FILE)
test_pf = pq.ParquetFile(TEST_FILE)
print("Train:", train_pf.metadata.num_rows)
print("Dev:", dev_pf.metadata.num_rows)
print("Test:", test_pf.metadata.num_rows)
assert train_pf.metadata.num_rows == 998000
assert dev_pf.metadata.num_rows == 1000
assert test_pf.metadata.num_rows == 1000


Train: 998000
Dev: 1000
Test: 1000


## 5. Prepare sentences


In [5]:
def prepare_sentence(sentence):
    return ["<START>"] + tokenize(sentence) + ["<END>"]


## 6. Build vocabulary from training data only


In [6]:
vocab = {"<START>", "<END>", "<UNK>"}
for batch in train_pf.iter_batches(batch_size=10000, columns=["text"]):
    for sentence in batch["text"].to_pylist():
        vocab.update(tokenize(sentence))
V = len(vocab)
print("Vocabulary size V:", V)
token_to_id = {t:i for i,t in enumerate(vocab)}
START_ID = token_to_id["<START>"]
END_ID = token_to_id["<END>"]
UNK_ID = token_to_id["<UNK>"]


Vocabulary size V: 593723


## 7. Build four n-gram counts


In [7]:
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()
quadrigram_counts = Counter()
processed = 0
for batch in train_pf.iter_batches(batch_size=10000, columns=["text"]):
    for sentence in batch["text"].to_pylist():
        ids = [token_to_id.get(t, UNK_ID) for t in prepare_sentence(sentence)]
        unigram_counts.update(ids)
        bigram_counts.update(zip(ids, ids[1:]))
        trigram_counts.update(zip(ids, ids[1:], ids[2:]))
        quadrigram_counts.update(zip(ids, ids[1:], ids[2:], ids[3:]))
        processed += 1
    if processed % 100000 == 0:
        print("Processed", processed)
print("Final processed:", processed)
assert processed == 998000
total_tokens = sum(unigram_counts.values())
print("Unique unigrams:", len(unigram_counts))
print("Unique bigrams:", len(bigram_counts))
print("Unique trigrams:", len(trigram_counts))
print("Unique quadrigrams:", len(quadrigram_counts))


Processed 100000
Processed 200000
Processed 300000
Processed 400000
Processed 500000
Processed 600000
Processed 700000
Processed 800000
Processed 900000
Final processed: 998000
Unique unigrams: 593722
Unique bigrams: 4813387
Unique trigrams: 8244124
Unique quadrigrams: 9197704


## 8. Add-K probability functions


In [8]:
def unigram_add_k_probability(w):
    return (unigram_counts[w] + K) / (total_tokens + K * V)

def bigram_add_k_probability(w1, w2):
    return (bigram_counts[(w1,w2)] + K) / (unigram_counts[w1] + K * V)

def trigram_add_k_probability(w1, w2, w3):
    return (trigram_counts[(w1,w2,w3)] + K) / (bigram_counts[(w1,w2)] + K * V)

def quadrigram_add_k_probability(w1, w2, w3, w4):
    return (quadrigram_counts[(w1,w2,w3,w4)] + K) / (trigram_counts[(w1,w2,w3)] + K * V)


## 9. Convert sentences to IDs


In [9]:
def sentence_to_ids(sentence):
    return [token_to_id.get(t, UNK_ID) for t in prepare_sentence(sentence)]


## 10. Add-K perplexity


In [10]:
def calculate_add_k_perplexity(sentences, model):
    total_log_probability = 0.0
    total_predictions = 0
    for sentence in sentences:
        ids = sentence_to_ids(sentence)
        if model == "unigram":
            for w in ids:
                total_log_probability += math.log(unigram_add_k_probability(w))
                total_predictions += 1
        elif model == "bigram":
            for i in range(1, len(ids)):
                total_log_probability += math.log(bigram_add_k_probability(ids[i-1], ids[i]))
                total_predictions += 1
        elif model == "trigram":
            for i in range(2, len(ids)):
                total_log_probability += math.log(trigram_add_k_probability(ids[i-2], ids[i-1], ids[i]))
                total_predictions += 1
        elif model == "quadrigram":
            for i in range(3, len(ids)):
                total_log_probability += math.log(quadrigram_add_k_probability(ids[i-3], ids[i-2], ids[i-1], ids[i]))
                total_predictions += 1
        else:
            raise ValueError("Invalid model")
    return math.exp(-total_log_probability / total_predictions)


## 11. Load development and test sets


In [11]:
dev_sentences = dev_pf.read_row_group(0)["text"].to_pylist()
test_sentences = test_pf.read_row_group(0)["text"].to_pylist()
assert len(dev_sentences) == 1000
assert len(test_sentences) == 1000
print("Development:", len(dev_sentences))
print("Test:", len(test_sentences))


Development: 1000
Test: 1000


## 12. Development / Validation results


In [12]:
dev_add_k_unigram = calculate_add_k_perplexity(dev_sentences, "unigram")
dev_add_k_bigram = calculate_add_k_perplexity(dev_sentences, "bigram")
dev_add_k_trigram = calculate_add_k_perplexity(dev_sentences, "trigram")
dev_add_k_quadrigram = calculate_add_k_perplexity(dev_sentences, "quadrigram")
print("Unigram:", dev_add_k_unigram)
print("Bigram:", dev_add_k_bigram)
print("Trigram:", dev_add_k_trigram)
print("Quadrigram:", dev_add_k_quadrigram)


Unigram: 3671.434061482684
Bigram: 12981.252223300247
Trigram: 120294.17664909312
Quadrigram: 301374.6526042435


## 13. Test results


In [13]:
test_add_k_unigram = calculate_add_k_perplexity(test_sentences, "unigram")
test_add_k_bigram = calculate_add_k_perplexity(test_sentences, "bigram")
test_add_k_trigram = calculate_add_k_perplexity(test_sentences, "trigram")
test_add_k_quadrigram = calculate_add_k_perplexity(test_sentences, "quadrigram")
print("Unigram:", test_add_k_unigram)
print("Bigram:", test_add_k_bigram)
print("Trigram:", test_add_k_trigram)
print("Quadrigram:", test_add_k_quadrigram)


Unigram: 4022.0497091211246
Bigram: 14591.42648019511
Trigram: 132181.5923501705
Quadrigram: 324671.91022525495


## 14. Final Add-K comparison


In [14]:
add_k_results = pd.DataFrame({
    "Model": ["Unigram", "Bigram", "Trigram", "Quadrigram"],
    "K": [K]*4,
    "Development Perplexity": [dev_add_k_unigram, dev_add_k_bigram, dev_add_k_trigram, dev_add_k_quadrigram],
    "Test Perplexity": [test_add_k_unigram, test_add_k_bigram, test_add_k_trigram, test_add_k_quadrigram]
})
display(add_k_results)
add_k_results.to_csv("add_k_results_k_0.3.csv", index=False)
print("Saved: add_k_results_k_0.3.csv")


,Model,K,Development Perplexity,Test Perplexity
0,Unigram,0.3,3671.434061,4022.049709
1,Bigram,0.3,12981.252223,14591.426480
2,Trigram,0.3,120294.176649,132181.592350
3,Quadrigram,0.3,301374.652604,324671.910225


Saved: add_k_results_k_0.3.csv


## 15. Conclusion

Add-K smoothing with K=0.3 was applied to all four n-gram language models. The models were evaluated on 1,000 development/validation sentences and 1,000 test sentences. Lower perplexity indicates better predictive performance.


## 16. Interpolated Smoothing

Interpolation combines multiple n-gram orders instead of relying only on the highest-order model. We use fixed weights that sum to 1.

In [15]:

# Interpolation weights.
# They can be changed and tuned on the development set.
LAMBDA_BIGRAM = (0.4, 0.6)                 # unigram, bigram
LAMBDA_TRIGRAM = (0.2, 0.3, 0.5)          # unigram, bigram, trigram
LAMBDA_QUADRIGRAM = (0.1, 0.2, 0.3, 0.4) # unigram, bigram, trigram, quadrigram

assert abs(sum(LAMBDA_BIGRAM) - 1.0) < 1e-9
assert abs(sum(LAMBDA_TRIGRAM) - 1.0) < 1e-9
assert abs(sum(LAMBDA_QUADRIGRAM) - 1.0) < 1e-9

def interpolated_bigram_probability(w1, w2):
    return (
        LAMBDA_BIGRAM[0] * unigram_add_k_probability(w2)
        + LAMBDA_BIGRAM[1] * bigram_add_k_probability(w1, w2)
    )

def interpolated_trigram_probability(w1, w2, w3):
    return (
        LAMBDA_TRIGRAM[0] * unigram_add_k_probability(w3)
        + LAMBDA_TRIGRAM[1] * bigram_add_k_probability(w2, w3)
        + LAMBDA_TRIGRAM[2] * trigram_add_k_probability(w1, w2, w3)
    )

def interpolated_quadrigram_probability(w1, w2, w3, w4):
    return (
        LAMBDA_QUADRIGRAM[0] * unigram_add_k_probability(w4)
        + LAMBDA_QUADRIGRAM[1] * bigram_add_k_probability(w3, w4)
        + LAMBDA_QUADRIGRAM[2] * trigram_add_k_probability(w2, w3, w4)
        + LAMBDA_QUADRIGRAM[3] * quadrigram_add_k_probability(w1, w2, w3, w4)
    )


In [16]:

def calculate_interpolated_perplexity(sentences, model):
    total_log_probability = 0.0
    total_predictions = 0

    for sentence in sentences:
        ids = sentence_to_ids(sentence)

        if model == "bigram":
            for i in range(1, len(ids)):
                p = interpolated_bigram_probability(ids[i-1], ids[i])

        elif model == "trigram":
            for i in range(2, len(ids)):
                p = interpolated_trigram_probability(ids[i-2], ids[i-1], ids[i])

        elif model == "quadrigram":
            for i in range(3, len(ids)):
                p = interpolated_quadrigram_probability(
                    ids[i-3], ids[i-2], ids[i-1], ids[i]
                )

        else:
            raise ValueError("Interpolated smoothing is defined here for bigram, trigram and quadrigram.")

        # The loops above must contribute probabilities. Recompute using a unified branch
        # so the function remains easy to read and avoids storing sentence probabilities.
        if model == "bigram":
            positions = range(1, len(ids))
            probs = (interpolated_bigram_probability(ids[i-1], ids[i]) for i in positions)
        elif model == "trigram":
            positions = range(2, len(ids))
            probs = (interpolated_trigram_probability(ids[i-2], ids[i-1], ids[i]) for i in positions)
        else:
            positions = range(3, len(ids))
            probs = (interpolated_quadrigram_probability(ids[i-3], ids[i-2], ids[i-1], ids[i]) for i in positions)

        for p in probs:
            total_log_probability += math.log(max(p, 1e-300))
            total_predictions += 1

    return math.exp(-total_log_probability / total_predictions)

dev_interp_bigram = calculate_interpolated_perplexity(dev_sentences, "bigram")
dev_interp_trigram = calculate_interpolated_perplexity(dev_sentences, "trigram")
dev_interp_quadrigram = calculate_interpolated_perplexity(dev_sentences, "quadrigram")

test_interp_bigram = calculate_interpolated_perplexity(test_sentences, "bigram")
test_interp_trigram = calculate_interpolated_perplexity(test_sentences, "trigram")
test_interp_quadrigram = calculate_interpolated_perplexity(test_sentences, "quadrigram")

print("Development:", dev_interp_bigram, dev_interp_trigram, dev_interp_quadrigram)
print("Test:", test_interp_bigram, test_interp_trigram, test_interp_quadrigram)


Development: 4276.687261576819 6560.728161760206 9166.430144710452
Test: 4728.15181995701 7355.719184763043 10456.608652239862


## 17. Good-Turing Smoothing

Good-Turing replaces an observed count \(c\) by an adjusted count \(c^*=(c+1)N_{c+1}/N_c\). For counts where the next frequency is unavailable, the original count is retained. Unseen events receive probability mass based on \(N_1/N\).

In [17]:

def frequency_of_frequencies(counts):
    return Counter(counts.values())

FOF = {
    "unigram": frequency_of_frequencies(unigram_counts),
    "bigram": frequency_of_frequencies(bigram_counts),
    "trigram": frequency_of_frequencies(trigram_counts),
    "quadrigram": frequency_of_frequencies(quadrigram_counts),
}

def gt_adjusted_count(c, fof, switch=5):
    if c == 0:
        return None
    Nc = fof.get(c, 0)
    Nc1 = fof.get(c + 1, 0)
    if c <= switch and Nc > 0 and Nc1 > 0:
        return (c + 1) * Nc1 / Nc
    return float(c)

def gt_zero_probability(fof, total_observed):
    # Classical Good-Turing estimate for an unseen event.
    N1 = fof.get(1, 0)
    if total_observed <= 0:
        return 0.0
    return N1 / total_observed

def gt_unigram_probability(w):
    c = unigram_counts[w]
    fof = FOF["unigram"]
    if c == 0:
        p0 = gt_zero_probability(fof, total_tokens)
        return p0 / max(V - len(unigram_counts), 1)
    c_star = gt_adjusted_count(c, fof)
    return c_star / total_tokens

# For conditional n-grams, normalize adjusted counts inside each context.
# Unseen continuations receive the Good-Turing zero mass.
def _gt_conditional_probability(context, word, ngram_counts, lower_context_counts, fof):
    c = ngram_counts[context + (word,)]
    context_count = lower_context_counts[context]
    if context_count == 0:
        return 0.0

    # Count how many distinct continuations have been observed after this context.
    observed_types = 0
    adjusted_sum = 0.0
    for ng, count in ngram_counts.items():
        if ng[:-1] == context:
            observed_types += 1
            adjusted_sum += gt_adjusted_count(count, fof)

    unseen_types = max(V - observed_types, 0)
    p0 = gt_zero_probability(fof, sum(ngram_counts.values()))
    denominator = adjusted_sum + unseen_types * p0 * context_count

    if denominator <= 0:
        return 0.0

    if c == 0:
        return (p0 * context_count) / denominator

    return gt_adjusted_count(c, fof) / denominator

# Cache context-level normalization to avoid repeatedly scanning all n-grams.
from functools import lru_cache

def _make_gt_context_cache(ngram_counts, lower_context_counts, fof):
    cache = {}
    totals = Counter()
    types = Counter()
    for ng, c in ngram_counts.items():
        context = ng[:-1]
        totals[context] += gt_adjusted_count(c, fof)
        types[context] += 1

    p0 = gt_zero_probability(fof, sum(ngram_counts.values()))
    for context, adjusted_sum in totals.items():
        unseen_types = max(V - types[context], 0)
        cache[context] = (adjusted_sum, unseen_types, p0)

    return cache

GT_CACHE = {
    "bigram": _make_gt_context_cache(bigram_counts, unigram_counts, FOF["bigram"]),
    "trigram": _make_gt_context_cache(trigram_counts, bigram_counts, FOF["trigram"]),
    "quadrigram": _make_gt_context_cache(quadrigram_counts, trigram_counts, FOF["quadrigram"]),
}

def gt_conditional_probability(context, word, order):
    if order == 2:
        counts, lower, fof = bigram_counts, unigram_counts, FOF["bigram"]
    elif order == 3:
        counts, lower, fof = trigram_counts, bigram_counts, FOF["trigram"]
    elif order == 4:
        counts, lower, fof = quadrigram_counts, trigram_counts, FOF["quadrigram"]
    else:
        raise ValueError("order must be 2, 3, or 4")

    context_count = lower[context]
    if context_count == 0:
        return 0.0

    adjusted_sum, unseen_types, p0 = GT_CACHE[{2:"bigram",3:"trigram",4:"quadrigram"}[order]][context]
    denom = adjusted_sum + unseen_types * p0 * context_count
    c = counts[context + (word,)]

    if denom <= 0:
        return 0.0
    if c == 0:
        return (p0 * context_count) / denom
    return gt_adjusted_count(c, fof) / denom

def calculate_good_turing_perplexity(sentences, model):
    total_log_probability, total_predictions = 0.0, 0
    for sentence in sentences:
        ids = sentence_to_ids(sentence)
        if model == "unigram":
            probs = (gt_unigram_probability(w) for w in ids)
        elif model == "bigram":
            probs = (gt_conditional_probability((ids[i-1],), ids[i], 2) for i in range(1, len(ids)))
        elif model == "trigram":
            probs = (gt_conditional_probability((ids[i-2], ids[i-1]), ids[i], 3) for i in range(2, len(ids)))
        elif model == "quadrigram":
            probs = (gt_conditional_probability((ids[i-3], ids[i-2], ids[i-1]), ids[i], 4) for i in range(3, len(ids)))
        else:
            raise ValueError("Invalid model")

        for p in probs:
            total_log_probability += math.log(max(p, 1e-300))
            total_predictions += 1
    return math.exp(-total_log_probability / total_predictions)


## 18. Katz Backoff Smoothing

Katz discounts low-frequency observed n-grams using Good-Turing adjusted counts. If an n-gram is unseen, the model backs off to the lower-order model using a context-specific normalization factor.

In [18]:

KATZ_CUTOFF = 5

def _build_katz_discounts(fof, cutoff=KATZ_CUTOFF):
    discounts = {}
    for c in range(1, cutoff + 1):
        c_star = gt_adjusted_count(c, fof)
        discounts[c] = (c_star / c) if c_star is not None else 1.0
    return discounts

KATZ_DISCOUNTS = {
    "bigram": _build_katz_discounts(FOF["bigram"]),
    "trigram": _build_katz_discounts(FOF["trigram"]),
    "quadrigram": _build_katz_discounts(FOF["quadrigram"]),
}

# Precompute Katz alpha(context) values once per order.
# This avoids scanning the complete n-gram dictionary for every test sentence.
def _build_katz_alpha_bigram():
    seen_lower = Counter()
    seen_discounted = Counter()

    for (w1,w2), c in bigram_counts.items():
        d = KATZ_DISCOUNTS["bigram"].get(c, 1.0)
        p_ml = c / max(unigram_counts[w1], 1)
        seen_discounted[w1] += d * p_ml
        seen_lower[w1] += gt_unigram_probability(w2)

    alpha = {}
    for w1 in unigram_counts:
        rem = max(1.0 - seen_discounted[w1], 1e-300)
        lower_unseen = max(1.0 - seen_lower[w1], 1e-300)
        alpha[w1] = rem / lower_unseen
    return alpha

KATZ_ALPHA_BIGRAM = _build_katz_alpha_bigram()

@lru_cache(maxsize=None)
def katz_bigram_probability(w1, w2):
    c = bigram_counts[(w1,w2)]
    denom = unigram_counts[w1]

    if denom == 0:
        return gt_unigram_probability(w2)

    if c > 0:
        d = KATZ_DISCOUNTS["bigram"].get(c, 1.0)
        return d * c / denom

    return KATZ_ALPHA_BIGRAM.get(w1, 1.0) * gt_unigram_probability(w2)


def _build_katz_alpha_trigram():
    seen_lower = Counter()
    seen_discounted = Counter()

    for (w1,w2,w3), c in trigram_counts.items():
        context = (w1,w2)
        d = KATZ_DISCOUNTS["trigram"].get(c, 1.0)
        p_ml = c / max(bigram_counts[context], 1)
        seen_discounted[context] += d * p_ml
        seen_lower[context] += katz_bigram_probability(w2,w3)

    alpha = {}
    for context in bigram_counts:
        rem = max(1.0 - seen_discounted[context], 1e-300)
        lower_unseen = max(1.0 - seen_lower[context], 1e-300)
        alpha[context] = rem / lower_unseen
    return alpha

KATZ_ALPHA_TRIGRAM = _build_katz_alpha_trigram()

@lru_cache(maxsize=None)
def katz_trigram_probability(w1,w2,w3):
    c = trigram_counts[(w1,w2,w3)]
    denom = bigram_counts[(w1,w2)]

    if denom == 0:
        return katz_bigram_probability(w2,w3)

    if c > 0:
        d = KATZ_DISCOUNTS["trigram"].get(c, 1.0)
        return d * c / denom

    return KATZ_ALPHA_TRIGRAM.get((w1,w2), 1.0) * katz_bigram_probability(w2,w3)


def _build_katz_alpha_quadrigram():
    seen_lower = Counter()
    seen_discounted = Counter()

    for (w1,w2,w3,w4), c in quadrigram_counts.items():
        context = (w1,w2,w3)
        d = KATZ_DISCOUNTS["quadrigram"].get(c, 1.0)
        p_ml = c / max(trigram_counts[context], 1)
        seen_discounted[context] += d * p_ml
        seen_lower[context] += katz_trigram_probability(w2,w3,w4)

    alpha = {}
    for context in trigram_counts:
        rem = max(1.0 - seen_discounted[context], 1e-300)
        lower_unseen = max(1.0 - seen_lower[context], 1e-300)
        alpha[context] = rem / lower_unseen
    return alpha

KATZ_ALPHA_QUADRIGRAM = _build_katz_alpha_quadrigram()

@lru_cache(maxsize=None)
def katz_quadrigram_probability(w1,w2,w3,w4):
    c = quadrigram_counts[(w1,w2,w3,w4)]
    denom = trigram_counts[(w1,w2,w3)]

    if denom == 0:
        return katz_trigram_probability(w2,w3,w4)

    if c > 0:
        d = KATZ_DISCOUNTS["quadrigram"].get(c, 1.0)
        return d * c / denom

    return KATZ_ALPHA_QUADRIGRAM.get((w1,w2,w3), 1.0) * katz_trigram_probability(w2,w3,w4)

def calculate_katz_perplexity(sentences, model):
    total_log_probability, total_predictions = 0.0, 0
    for sentence in sentences:
        ids = sentence_to_ids(sentence)
        if model == "bigram":
            probs = (katz_bigram_probability(ids[i-1],ids[i]) for i in range(1,len(ids)))
        elif model == "trigram":
            probs = (katz_trigram_probability(ids[i-2],ids[i-1],ids[i]) for i in range(2,len(ids)))
        elif model == "quadrigram":
            probs = (katz_quadrigram_probability(ids[i-3],ids[i-2],ids[i-1],ids[i]) for i in range(3,len(ids)))
        else:
            raise ValueError("Katz is evaluated here for bigram, trigram and quadrigram.")

        for p in probs:
            total_log_probability += math.log(max(p,1e-300))
            total_predictions += 1

    return math.exp(-total_log_probability/total_predictions)


## 19. Stupid Backoff Smoothing

Stupid Backoff uses the maximum-likelihood probability when an n-gram is observed. When it is unseen, it backs off to the lower-order model multiplied by a fixed factor \(\alpha=0.4\). It is a scoring method rather than a normalized probability model, so perplexity here is a comparable log-score/perplexity-style metric.

In [19]:

STUPID_BACKOFF_ALPHA = 0.4

def stupid_backoff_bigram(w1,w2):
    if bigram_counts[(w1,w2)] > 0:
        return bigram_counts[(w1,w2)] / unigram_counts[w1]
    return STUPID_BACKOFF_ALPHA * unigram_counts[w2] / max(total_tokens,1)

def stupid_backoff_trigram(w1,w2,w3):
    if trigram_counts[(w1,w2,w3)] > 0:
        return trigram_counts[(w1,w2,w3)] / bigram_counts[(w1,w2)]
    return STUPID_BACKOFF_ALPHA * stupid_backoff_bigram(w2,w3)

def stupid_backoff_quadrigram(w1,w2,w3,w4):
    if quadrigram_counts[(w1,w2,w3,w4)] > 0:
        return quadrigram_counts[(w1,w2,w3,w4)] / trigram_counts[(w1,w2,w3)]
    return STUPID_BACKOFF_ALPHA * stupid_backoff_trigram(w2,w3,w4)

def calculate_stupid_backoff_perplexity(sentences, model):
    total_log_score, total_predictions = 0.0, 0
    for sentence in sentences:
        ids = sentence_to_ids(sentence)
        if model == "bigram":
            probs = (stupid_backoff_bigram(ids[i-1],ids[i]) for i in range(1,len(ids)))
        elif model == "trigram":
            probs = (stupid_backoff_trigram(ids[i-2],ids[i-1],ids[i]) for i in range(2,len(ids)))
        elif model == "quadrigram":
            probs = (stupid_backoff_quadrigram(ids[i-3],ids[i-2],ids[i-1],ids[i]) for i in range(3,len(ids)))
        else:
            raise ValueError("Stupid Backoff is evaluated here for bigram, trigram and quadrigram.")
        for p in probs:
            total_log_score += math.log(max(p,1e-300))
            total_predictions += 1
    return math.exp(-total_log_score/total_predictions)


## 20. Kneser-Ney Smoothing

Kneser-Ney uses continuation probabilities. For bigrams, the lower-order probability of a word depends on the number of distinct contexts in which it appears. Higher-order models recursively use lower-order Kneser-Ney probabilities. We use absolute discounting with a development-tunable discount \(D=0.75\).

In [20]:

KN_D = 0.75

# Distinct continuation/context statistics.
bigram_context_types = Counter()
bigram_continuation_types = Counter()
trigram_context_types = Counter()
quadrigram_context_types = Counter()

for (w1,w2) in bigram_counts:
    bigram_context_types[w1] += 1
    bigram_continuation_types[w2] += 1

for (w1,w2,w3) in trigram_counts:
    trigram_context_types[(w1,w2)] += 1

for (w1,w2,w3,w4) in quadrigram_counts:
    quadrigram_context_types[(w1,w2,w3)] += 1

num_bigram_types = len(bigram_counts)

@lru_cache(maxsize=None)
def kn_unigram(w):
    return bigram_continuation_types[w] / max(num_bigram_types,1)

@lru_cache(maxsize=None)
def kn_bigram(w1,w2):
    c = bigram_counts[(w1,w2)]
    context_count = unigram_counts[w1]
    if context_count == 0:
        return kn_unigram(w2)

    first = max(c - KN_D, 0.0) / context_count
    lambda_h = KN_D * bigram_context_types[w1] / context_count
    return first + lambda_h * kn_unigram(w2)

@lru_cache(maxsize=None)
def kn_trigram(w1,w2,w3):
    c = trigram_counts[(w1,w2,w3)]
    context_count = bigram_counts[(w1,w2)]
    if context_count == 0:
        return kn_bigram(w2,w3)

    first = max(c - KN_D, 0.0) / context_count
    lambda_h = KN_D * trigram_context_types[(w1,w2)] / context_count
    return first + lambda_h * kn_bigram(w2,w3)

@lru_cache(maxsize=None)
def kn_quadrigram(w1,w2,w3,w4):
    c = quadrigram_counts[(w1,w2,w3,w4)]
    context_count = trigram_counts[(w1,w2,w3)]
    if context_count == 0:
        return kn_trigram(w2,w3,w4)

    first = max(c - KN_D, 0.0) / context_count
    lambda_h = KN_D * quadrigram_context_types[(w1,w2,w3)] / context_count
    return first + lambda_h * kn_trigram(w2,w3,w4)

def calculate_kneser_ney_perplexity(sentences, model):
    total_log_probability, total_predictions = 0.0, 0
    for sentence in sentences:
        ids = sentence_to_ids(sentence)
        if model == "bigram":
            probs = (kn_bigram(ids[i-1],ids[i]) for i in range(1,len(ids)))
        elif model == "trigram":
            probs = (kn_trigram(ids[i-2],ids[i-1],ids[i]) for i in range(2,len(ids)))
        elif model == "quadrigram":
            probs = (kn_quadrigram(ids[i-3],ids[i-2],ids[i-1],ids[i]) for i in range(3,len(ids)))
        else:
            raise ValueError("Kneser-Ney is evaluated here for bigram, trigram and quadrigram.")
        for p in probs:
            total_log_probability += math.log(max(p,1e-300))
            total_predictions += 1
    return math.exp(-total_log_probability/total_predictions)


## 21. Evaluate all smoothing techniques

All methods are evaluated on the same development and test sentences. Unigram is retained for Good-Turing, while the other smoothing methods are designed for bigram, trigram and quadrigram models.

In [21]:

# Calculate results. Depending on hardware, Katz can be the slowest method
# because its backoff normalization searches observed continuations.
results = []

def add_result(method, model, dev_ppl, test_ppl):
    results.append({
        "Smoothing": method,
        "N-gram Model": model,
        "Development Perplexity": dev_ppl,
        "Test Perplexity": test_ppl
    })

# Interpolated
add_result("Interpolated", "Bigram", dev_interp_bigram, test_interp_bigram)
add_result("Interpolated", "Trigram", dev_interp_trigram, test_interp_trigram)
add_result("Interpolated", "Quadrigram", dev_interp_quadrigram, test_interp_quadrigram)

# Good-Turing
for model in ["unigram", "bigram", "trigram", "quadrigram"]:
    d = calculate_good_turing_perplexity(dev_sentences, model)
    t = calculate_good_turing_perplexity(test_sentences, model)
    add_result("Good-Turing", model.capitalize(), d, t)

# Katz
for model in ["bigram", "trigram", "quadrigram"]:
    d = calculate_katz_perplexity(dev_sentences, model)
    t = calculate_katz_perplexity(test_sentences, model)
    add_result("Katz Backoff", model.capitalize(), d, t)

# Stupid Backoff
for model in ["bigram", "trigram", "quadrigram"]:
    d = calculate_stupid_backoff_perplexity(dev_sentences, model)
    t = calculate_stupid_backoff_perplexity(test_sentences, model)
    add_result("Stupid Backoff", model.capitalize(), d, t)

# Kneser-Ney
for model in ["bigram", "trigram", "quadrigram"]:
    d = calculate_kneser_ney_perplexity(dev_sentences, model)
    t = calculate_kneser_ney_perplexity(test_sentences, model)
    add_result("Kneser-Ney", model.capitalize(), d, t)

smoothing_results = pd.DataFrame(results)
display(smoothing_results.sort_values(["N-gram Model", "Test Perplexity"]))
smoothing_results.to_csv("all_smoothing_results.csv", index=False)
print("Saved: all_smoothing_results.csv")


,Smoothing,N-gram Model,Development Perplexity,Test Perplexity
7,Katz Backoff,Bigram,6.290974e+02,7.101759e+02
0,Interpolated,Bigram,4.276687e+03,4.728152e+03
13,Kneser-Ney,Bigram,5.717809e+11,2.013991e+11
10,Stupid Backoff,Bigram,5.812136e+11,2.064676e+11
4,Good-Turing,Bigram,1.000000e+300,1.000000e+300
9,Katz Backoff,Quadrigram,6.261115e+02,9.071319e+02
2,Interpolated,Quadrigram,9.166430e+03,1.045661e+04
15,Kneser-Ney,Quadrigram,4.556199e+11,5.569597e+11
12,Stupid Backoff,Quadrigram,1.378182e+12,1.742059e+12
6,Good-Turing,Quadrigram,2.450968e+214,7.339651e+216


Saved: all_smoothing_results.csv


## 22. Complete comparison with Add-K

This table combines the previous Add-K results with the new smoothing methods. Use the development set to tune hyperparameters such as interpolation weights, Katz cutoff, Stupid Backoff alpha, and Kneser-Ney discount.

In [22]:

# Add-K results already computed above.
comparison = pd.concat([
    add_k_results.assign(Smoothing="Add-K"),
    smoothing_results.rename(columns={
        "N-gram Model": "Model"
    })[["Smoothing","Model","Development Perplexity","Test Perplexity"]]
], ignore_index=True, sort=False)

display(comparison.sort_values(["Model","Test Perplexity"]))
comparison.to_csv("complete_smoothing_comparison.csv", index=False)
print("Saved: complete_smoothing_comparison.csv")


,Model,K,Development Perplexity,Test Perplexity,Smoothing
11,Bigram,NaN,6.290974e+02,7.101759e+02,Katz Backoff
4,Bigram,NaN,4.276687e+03,4.728152e+03,Interpolated
1,Bigram,0.3,1.298125e+04,1.459143e+04,Add-K
17,Bigram,NaN,5.717809e+11,2.013991e+11,Kneser-Ney
14,Bigram,NaN,5.812136e+11,2.064676e+11,Stupid Backoff
8,Bigram,NaN,1.000000e+300,1.000000e+300,Good-Turing
13,Quadrigram,NaN,6.261115e+02,9.071319e+02,Katz Backoff
6,Quadrigram,NaN,9.166430e+03,1.045661e+04,Interpolated
3,Quadrigram,0.3,3.013747e+05,3.246719e+05,Add-K
19,Quadrigram,NaN,4.556199e+11,5.569597e+11,Kneser-Ney


Saved: complete_smoothing_comparison.csv


## 23. Interpretation / Viva Points

**Interpolated smoothing:** combines lower- and higher-order models, so information from shorter histories is still available.

**Good-Turing:** estimates the probability of unseen and rare events using the frequency-of-frequencies \(N_c\).

**Katz Backoff:** discounts seen n-grams and backs off to a lower-order model when an n-gram is unseen.

**Stupid Backoff:** uses maximum-likelihood probabilities for seen n-grams and multiplies the lower-order score by a fixed factor (0.4) when backing off. It is not normalized like a true probability model.

**Kneser-Ney:** improves the lower-order distribution using continuation counts rather than raw word frequency; this is especially effective for language modeling.

**Evaluation:** perplexity is \(\exp(-\frac{1}{N}\sum \log p(w_i|h_i))\). Lower perplexity is better. Development/validation data should be used for tuning parameters, while the test set should be used only for final comparison.
